# 01 - Project Overview

Author: Saige Mukherjee

Contact: mukherjeesaige@gmail.com //
https://www.linkedin.com/in/saige-mukherjee-0aba68281/

This notebook contains no code.

# Hospital Patient Flow Analysis with MIMIC-IV

This project analyzes patient movement through Beth Israel Deaconess Medical Center using the MIMIC-IV clinical database. The focus is operational rather than clinical prediction: identifying where hospital stays and transfers consume the most time, where long-tail delays appear, and which care units contribute most to excess bed-hours.

The project is designed as a portfolio-quality healthcare operations analysis using SQL, Python, and reproducible notebooks.

## Project motivation

Hospitals operate under capacity pressure. Even when clinical care is appropriate, patient flow can be affected by bed availability, care-unit bottlenecks, discharge timing, transfer delays, and high-variance patient pathways.

This project asks:

1. Which care units account for the largest share of inpatient time?
2. Where do long stays cluster?
3. Which units generate the most excess bed-hours above operational thresholds such as 48 or 72 hours?
4. How do broad care-unit groups differ from individual units?
5. Which parts of the hospital appear to create high operational burden because of volume, duration, or variability?

The goal is not to infer individual clinical appropriateness. The goal is to describe operational patterns at an aggregate level.

## Executive summary

This project analyzes hospital patient flow in MIMIC-IV using aggregate transfer-duration metrics.

Early results show that hospital flow burden is highly concentrated: a small number of care units account for a large share of excess bed-hours above operational thresholds such as 72 hours. The analysis separates high-volume system burden from low-volume long-tail outliers.

The main deliverables are:
- whole-hospital transfer-duration KPIs
- care-unit burden rankings
- excess bed-hour analysis
- final portfolio-ready summary figures
- an executive summary notebook for non-technical readers

## Dataset

This project uses **MIMIC-IV v3.1**, accessed through PhysioNet and Google BigQuery.

MIMIC-IV is a deidentified electronic health record dataset from Beth Israel Deaconess Medical Center. It includes hospital-wide data, ICU data, patient demographics, admissions, transfers, labs, procedures, diagnoses, medication records, and other clinical information.

For this project, the main tables are:

| Table        | Module | Purpose in this project                       |
| ------------ | -----: | --------------------------------------------- |
| `patients`   | `hosp` | Demographics and deidentified time anchoring  |
| `admissions` | `hosp` | Hospital admissions and discharge information |
| `transfers`  | `hosp` | Patient movement between care units           |
| `services`   | `hosp` | Hospital service assignment over time         |
| `icustays`   |  `icu` | ICU stay definitions derived from transfers   |

The core unit of analysis is usually a **transfer segment**: a period of time where a patient is documented in a specific care unit.


## Privacy and publication constraints

This repository does **not** publish MIMIC-IV data.

Because MIMIC-IV is a credentialed-access clinical dataset, this project only publishes:

* SQL queries
* Python analysis code
* aggregate tables
* aggregate figures
* methodology notes
* derived metrics with appropriate suppression rules

This project does **not** publish:

* raw MIMIC-IV rows
* patient-level extracts
* small-cell tables
* timestamps tied to individual patients
* any data that could reasonably support reidentification

For public outputs, aggregate results should use conservative minimum-cell-size checks. Small groups should be suppressed or combined into broader categories.

## BigQuery setup

The project assumes access to the PhysioNet BigQuery project.

Expected datasets:

```text
physionet-data.mimiciv_v3_1_hosp
physionet-data.mimiciv_v3_1_icu
```

The main hospital module is assigned to `hosp`, and the ICU module is assigned to `icu`.

## Core operational definitions

### Transfer segment

A transfer segment is one row from the `transfers` table with a valid `intime` and `outtime`.

```text
transfer_hours = outtime - intime
```

### Care unit

A care unit is the documented hospital location or level-of-care label in `transfers.careunit`.

Care-unit labels are treated as operational labels. They may represent physical units, service areas, levels of care, or documentation conventions depending on how the source systems recorded patient movement.

### Broad care-unit group

Individual care units are mapped into broader operational categories such as:

* Emergency Department
* Observation
* Medical Ward
* Surgical Ward
* ICU
* Cardiac
* Neurology
* Psychiatry
* Oncology
* Obstetrics / Gynecology
* Other / Unknown

These broad groups are used to reduce noise and make results easier to interpret.

## Main metrics

This project uses robust, operations-friendly metrics rather than relying only on averages.

| Metric                   | Meaning                              |
| ------------------------ | ------------------------------------ |
| `visits`                 | Number of transfer segments or stays |
| `median_hours`           | Typical duration                     |
| `p75_hours`              | Upper-middle duration                |
| `p90_hours`              | Long-stay threshold                  |
| `p95_hours`              | Severe long-stay threshold           |
| `p99_hours`              | Extreme long-tail duration           |
| `% over 48h`             | Share of segments longer than 2 days |
| `% over 72h`             | Share of segments longer than 3 days |
| `excess_hours_72h`       | Total hours above 72 hours           |
| `trimmed_mean_under_p99` | Mean after excluding extreme top 1%  |

The project emphasizes both **intensity** and **burden**:

* A small unit can have very long stays but limited system-wide burden.
* A large unit can have moderate stays but create major aggregate bed-hour burden.
* The most operationally important units often combine high volume, high duration, and high variability.

## Key analysis notebooks

Suggested notebook sequence:

| Notebook | Purpose |
|---|---|
| `00_executive_summary.ipynb` | Short, high-impact presentation of the main KPIs, visuals, findings, limitations, and operational takeaways |
| `01_project_overview.ipynb` | Project framing, dataset notes, privacy rules, operational definitions, and notebook structure |
| `02_data_inventory.ipynb` | Table checks, row counts, schema review, key identifiers, and date/window assumptions |
| `03_whole_hospital_flow.ipynb` | Whole-hospital transfer-time distribution, summary metrics, and overall patient-flow burden |
| `04_careunit_burden_overall.ipynb` | Comparison of care units and broader care-unit groups by overall transfer volume, duration, and bed-hour burden |
| `05_longstay_careunit_burden.ipynb` | Ranking of care units by long-stay and excess bed-hour burden, including threshold-based and cumulative-contribution analyses |
| `06_discharge_hypothesis.ipynb` | Investigation of whether prolonged final-care-unit stays and discharge destinations support a downstream-care constraint hypothesis |

The repository also includes `Executive_Summary.pdf`, a polished and shareable version of the executive summary for quick review.

## Example analysis pattern

Most notebooks follow this pattern:

1. Define the operational question.
2. Write a BigQuery SQL query.
3. Pull only the aggregate or analysis-ready result needed.
4. Check privacy constraints.
5. Visualize the result.
6. Interpret the result from a hospital operations perspective.
7. State limitations.

The preferred interpretation style is:

```text
What does this tell a hospital operations manager?
What decision could this inform?
What can this analysis not prove?

## Important limitations

This project cannot determine whether a long stay was clinically appropriate.

Long transfer durations may reflect many different causes, including:

* severity of illness
* planned monitoring
* bed availability
* staffing constraints
* delayed procedures
* discharge barriers
* documentation conventions
* administrative artifacts

The analysis can identify patterns, outliers, and operational burden. It cannot directly assign cause without additional clinical, staffing, scheduling, or qualitative data.

## Current project status

Completed or in progress:

* Accessed MIMIC-IV v3.1 through BigQuery
* Explored `transfers`, `admissions`, and care-unit labels
* Calculated whole-hospital transfer-duration metrics
* Built broad care-unit groupings
* Compared individual care units against broad groups
* Ranked care units by excess bed-hours above 72 hours
* Investigated ED Observation and transfer/admit event labeling

Next analytical priorities:

1. Finalize the care-unit mapping.
2. Produce a clean whole-hospital flow summary.
3. Create a ranked care-unit burden table.
4. Separate high-volume burden from low-volume extreme outliers.
5. Decide whether to integrate MIMIC-IV-ED for ED-specific questions.
6. Prepare aggregate figures for GitHub and portfolio use.

## Reproducibility notes

This repository is intended to be reproducible for users who already have credentialed MIMIC-IV access.

To reproduce the analysis:

1. Obtain MIMIC-IV access through PhysioNet.
2. Confirm access to the MIMIC-IV v3.1 BigQuery datasets.
3. Set your Google Cloud project ID.
4. Run notebooks in numeric order.
5. Do not export or publish row-level patient data.

Because the underlying dataset is credentialed, public users without MIMIC-IV access can review the code and aggregate outputs but cannot rerun the analysis directly.

## Citation

This project uses MIMIC-IV v3.1:

Johnson, A., Bulgarelli, L., Pollard, T., Gow, B., Moody, B., Horng, S., Celi, L. A., & Mark, R. (2024). MIMIC-IV (version 3.1). PhysioNet.

Original publication:

Johnson, A. E. W., Bulgarelli, L., Shen, L., et al. MIMIC-IV, a freely accessible electronic health record dataset. Scientific Data, 10, 1 (2023).


---